In [ ]:
#27.PID namespace

Explanation

#This difference happens because of PID namespaces in Docker.

#Inside the container, the nginx process runs as PID 1, which means it is the main process of that container.

#On the host, the same process has a completely different PID (e.g. 2302) because it is part of the host’s global process table.

#PowerShell Output:
PS C:\Users\awans> docker run -d --name isolated nginx:alpine
d035c8aa2aa8baa1c61d2e97ebe8b9ec513308eac45c2f715257a15989a6ed98
PS C:\Users\awans> docker top isolated
UID                 PID                 PPID                C                   STIME               TTY                 TIME                CMD
root                2302                2278                0                   04:08               ?                   00:00:00            nginx: master process nginx -g daemon off;
statd               2341                2302                0                   04:08               ?                   00:00:00            nginx: worker process
statd               2342                2302                0                   04:08               ?                   00:00:00            nginx: worker process
statd               2343                2302                0                   04:08               ?                   00:00:00            nginx: worker process
statd               2344                2302                0                   04:08               ?                   00:00:00            nginx: worker process
PS C:\Users\awans> docker exec -it isolated sh
/ # ps aux
PID   USER     TIME  COMMAND
    1 root      0:00 nginx: master process nginx -g daemon off;
   31 nginx     0:00 nginx: worker process
   32 nginx     0:00 nginx: worker process
   33 nginx     0:00 nginx: worker process
   34 nginx     0:00 nginx: worker process
   35 root      0:00 sh
   41 root      0:00 ps aux
/ #

In [ ]:
#28 cgroup limits

What happens?

#The Python process is terminated immediately when it tries to allocate more memory than the container’s limit.

Why does this happen?

#Docker uses cgroups (control groups) from the Linux kernel to enforce resource limits.

#The container is limited to 100MB of RAM

#The Python command tries to allocate 200MB

#This exceeds the allowed memory

#The Linux kernel triggers an Out Of Memory (OOM) kill

#The process is forcefully stopped → shown as "Killed"

#Powershell Output:
Monitor Usage:
CONTAINER ID   NAME      CPU %     MEM USAGE / LIMIT   MEM %     NET I/O       BLOCK I/O       PIDS
1616a9a7701b   limited   0.00%     592KiB / 100MiB     0.58%     872B / 126B   459kB / 105MB   2
PS C:\Users\awans> docker exec -it limited sh
# python3 -c "x = bytearray(200 * 1024 * 1024)"
Killed

In [ ]:
#29.Network Namespace

Record IP addresses
#Inside the container (ip addr):

#IP address: 172.17.0.4/16

#On the host (ipconfig):

#IP address: 192.168.0.23/24

Are they on the same subnet?

#No, they are not on the same subnet.

#Container: 172.17.0.0/16

#Host (Wi-Fi): 192.168.0.0/24

How Docker assigns IPs

#Docker runs a virtual network bridge

#Each container gets:

#Its own network namespace

#Its own virtual network interface (eth0)

#An IP from Docker’s internal range (172.17.x.x)

#Docker uses NAT (Network Address Translation) so containers can access the internet through the host

#Powershell Output:
PS C:\Users\awans> docker run -it --rm alpine sh
Unable to find image 'alpine:latest' locally
latest: Pulling from library/alpine
9e595aac14e0: Download complete
caa817ad3aea: Download complete
Digest: sha256:25109184c71bdad752c8312a8623239686a9a2071e8825f20acb8f2198c3f659
Status: Downloaded newer image for alpine:latest
/ # ip addr
1: lo: <LOOPBACK,UP,LOWER_UP> mtu 65536 qdisc noqueue state UNKNOWN qlen 1000
    link/loopback 00:00:00:00:00:00 brd 00:00:00:00:00:00
    inet 127.0.0.1/8 scope host lo
       valid_lft forever preferred_lft forever
    inet6 ::1/128 scope host
       valid_lft forever preferred_lft forever
2: eth0@if12: <BROADCAST,MULTICAST,UP,LOWER_UP,M-DOWN> mtu 1500 qdisc noqueue state UP
    link/ether c2:df:db:90:85:00 brd ff:ff:ff:ff:ff:ff
    inet 172.17.0.4/16 brd 172.17.255.255 scope global eth0
       valid_lft forever preferred_lft forever
/ # exit
PS C:\Users\awans> ipconfig

Windows IP Configuration


Wireless LAN adapter Local Area Connection* 1:

   Media State . . . . . . . . . . . : Media disconnected
   Connection-specific DNS Suffix  . :

Wireless LAN adapter Local Area Connection* 2:

   Media State . . . . . . . . . . . : Media disconnected
   Connection-specific DNS Suffix  . :

Wireless LAN adapter Wi-Fi:

   Connection-specific DNS Suffix  . : Home
   IPv6 Address. . . . . . . . . . . : 2a02:c7c:5168:9b00:d322:acdd:99b2:6b9a
   IPv6 Address. . . . . . . . . . . : fd50:10f8:641:0:8dd5:70b2:c4ee:8cfd
   Temporary IPv6 Address. . . . . . : 2a02:c7c:5168:9b00:3429:a979:68fb:5acb
   Temporary IPv6 Address. . . . . . : fd50:10f8:641:0:5591:68f1:854f:a0ad
   Link-local IPv6 Address . . . . . : fe80::56b0:2789:b763:1546%4
   IPv4 Address. . . . . . . . . . . : 192.168.0.23
   Subnet Mask . . . . . . . . . . . : 255.255.255.0
   Default Gateway . . . . . . . . . : fe80::3e45:7aff:fe3a:1d79%4
                                       192.168.0.1

Ethernet adapter Bluetooth Network Connection:

   Media State . . . . . . . . . . . : Media disconnected
   Connection-specific DNS Suffix  . :

Ethernet adapter vEthernet (WSL (Hyper-V firewall)):

   Connection-specific DNS Suffix  . :
   Link-local IPv6 Address . . . . . : fe80::440e:fa7:32b0:9464%35
   IPv4 Address. . . . . . . . . . . : 172.31.32.1
   Subnet Mask . . . . . . . . . . . : 255.255.240.0
   Default Gateway . . . . . . . . . :
PS C:\Users\awans>

In [ ]:
#30.Read-Only root

What error do you get?

#The error is:

#Read-only file system

#This means the container does not allow any write operations to its root filesystem.

Why use --read-only in production containers?

#Running containers as read-only improves security and stability:

#Prevents attackers from modifying files if the container is compromised

#Protects application files from accidental changes

#Ensures consistency — the container always runs in the same state

#Limits damage — malware can’t easily write to disk

#PowerShell Output:
PS C:\Users\awans> docker run --read-only -it --rm alpine sh
/ # touch /test.txt
touch: /test.txt: Read-only file system



In [ ]:
#31.Escape Attempt

#What do you see?

You can see the entire host filesystem from inside the container.

The /host directory is actually the root directory of the host machine (/)

This means the container has access to all host files, including system directories

#Why should this NEVER be used in production?

 Security risk — attackers could access the entire host system

 No isolation — defeats the purpose of containers

 System damage — files on the host could be modified or deleted

 Privilege escalation — container could act like root on the host

#PowerShell Output:

PS C:\Users\awans> docker run -v /:/host -it --rm alpine sh
/ # ls /host
EFI                       home                      mutagen-file-shares       services
bin                       host-network.o            mutagen-file-shares-mark  src
boot                      host_mnt                  opt                       srv
bpf-legacy.o              init                      parent-distro             sys
bpf.o                     initd                     proc                      tmp
containers                lib                       pwatch.o                  udpv6csum.o
dev                       lib64                     root                      usr
dpkg.orig                 media                     run                       var
etc                       mnt                       sbin